# Input Files and QuPath Annotation Guide

This notebook explains the input files required by HistoMapTx and how to prepare histological annotations using QuPath.

---

## 1. Required Input Files

HistoMapTx requires two main inputs:

| Input | Description |
|-------|-------------|
| **Visium SpatialData object** | Output from `spatialdata-io` reading of a 10x Visium Space Ranger folder |
| **GeoJSON annotation file** | Exported from QuPath, containing tissue region annotations |

### 1.1 Visium SpatialData Object

Load your Visium data using `spatialdata-io`:

```python
import spatialdata_io

sdata = spatialdata_io.visium("/path/to/spaceranger_output/")
```

The Space Ranger output folder should contain:
```
spaceranger_output/
├── filtered_feature_bc_matrix/
├── spatial/
│   ├── tissue_positions.csv
│   ├── scalefactors_json.json
│   ├── tissue_hires_image.png
│   └── tissue_lowres_image.png
└── ...
```

### 1.2 GeoJSON Annotation File

The GeoJSON file is exported from QuPath and contains polygon geometries for each annotated tissue region. See Section 2 below for how to generate it.

> **Note:** If your GeoJSON contains very large polygons, you may need to disable the OGR size limit:
> ```python
> import os
> os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"  # 0 = no limit
> ```

---

## 2. QuPath Annotation

### 2.1 Video Tutorial

A step-by-step video demonstration of the annotation workflow in QuPath is available here:

**[▶ Watch: QuPath Annotation Tutorial for HistoMapTx](https://www.youtube.com/watch?v=vr9w_LYtSso)**

And the annotation documentation of QuPath is available here:  

**[QuPath documentation](https://qupath.readthedocs.io/en/stable/docs/starting/annotating.html)**

---

### 2.2 Which Image to Annotate

> ⚠️ **Critical:** You must annotate the **full-resolution image** that was provided to Space Ranger — **not** the downscaled `tissue_hires_image.png` or `tissue_lowres_image.png` generated by Space Ranger.

Space Ranger generates scaled-down images for visualization purposes only. The spot coordinates in `tissue_positions.csv` are expressed in the coordinate system of the **original full-resolution image**. If you annotate a downscaled image, your polygon coordinates will not align with the spot positions.

**Use the original `.tif` / `.btf` / `.svs` file** (the same one you passed to `spaceranger count --image`).

---

### 2.3 Manual Annotation in QuPath

1. Open QuPath and load your full-resolution image (`File > Open`)
2. Select an annotation tool from the toolbar (brush, polygon, wand, etc.)
3. Draw regions of interest on the tissue
4. Double-click a completed annotation to name it (e.g. `Tumor`, `Stroma`, `Necrosis`) or set annotation class in the annotation pannel
5. Repeat for all regions
6. Export the annotations (see Section 2.5)

---

### 2.4 Pixel Thresholding for Tissue Detection (Optional)

If you want to automatically detect tissue regions rather than annotating manually, QuPath provides a built-in pixel classifier:

1. Go to **Analyze > Pixel classification > Create thresholder**
2. Choose a color transform (e.g. `Optical density sum` works well for H&E)
3. Adjust the threshold slider until tissue is cleanly separated from background
4. Set **Region** to `Full image` and **Output** to `Annotation`
5. Click **Create objects** — QuPath will generate annotation polygons for the detected tissue
6. Give the resulting annotation a name (e.g. `Tissue`)
7. Export the annotations (see Section 2.5)

This approach is useful for quickly delineating the full tissue area before adding finer manual annotations on top.

---

### 2.5 Exporting Annotations as GeoJSON

Once your annotations are ready:

1. Go to **File > Export as > GeoJSON**  
   *(or right-click in the Annotations panel > Export as GeoJSON)*
2. Make sure **"Export all objects"** is selected
3. Save as `annotations.geojson`

---

## 3. Loading into HistoMapTx

Once you have both files, create the HistoMap object:

In [ ]:
import os
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"  # required if annotations contain large polygons

import spatialdata_io
import histomaptx as hm

# Load Visium data
sdata = spatialdata_io.visium("/path/to/spaceranger_output/")

# Create HistoMap object
histomap = hm.HistoMap(
    visium_spatialdata=sdata,
    geojson_path="/path/to/annotations.geojson",
    full_res_path="/path/to/full_res_image.tif"
)

You are now ready to proceed to the next tutorial: **1. Tissue Detection**.